In [35]:
import numpy as np
import soundfile as sf

MARK_HZ = 2225
SPACE_HZ = 2025

BITRATE = 48000

BYTE_LENGTH_SAMPLES = 1600
BIT_LENGTH_SAMPLES = 160

In [36]:
# Tone power function
def tone_power(samples, N, f, fs):
    #print(samples)
    I = 0
    Q = 0
    for n in range(N):
        angle = 2 * np.pi * f * n / fs
        I     = I + samples[n] * np.cos(angle)
        Q     = Q + samples[n] * np.sin(angle)
    return (I ** 2) + (Q ** 2)

In [ ]:
# Import .wav files as arrays
message_data, message_fs = sf.read("message.wav", dtype=np.int16)
test1_data, test1_fs = sf.read("test1.wav", dtype=np.int16)
test2_data, test2_fs = sf.read("test2.wav", dtype=np.int16)

In [38]:
# Test tones
test_space = np.sin(2 * np.pi * SPACE_HZ * np.arange(BITRATE * 5) / BITRATE)
test_mark = np.sin(2 * np.pi * MARK_HZ * np.arange(BITRATE * 5) / BITRATE)

print(test_space[0:20])
print(test_mark[0:20])

[ 0.          0.26197864  0.50565737  0.7140146   0.87249601  0.97003125
  0.99980724  0.95974404  0.85264016  0.68597711  0.47139674  0.22388805
 -0.03925982 -0.29966528 -0.53913832 -0.74095113 -0.89100652 -0.97882275
 -0.99826561 -0.94797697]
[ 0.          0.28715155  0.55011642  0.76674516  0.91879121  0.99344778
  0.98442657  0.89248743  0.72537437  0.49716327  0.22707626 -0.0621373
 -0.34611706 -0.60094349 -0.80515265 -0.94154407 -0.99862953 -0.97160077
 -0.86273439 -0.68120017]


In [39]:
# Testing space detection
space_detections = 0
mark_detections = 0
for i in range(len(test_space) // BYTE_LENGTH_SAMPLES):
    block = test_space[(i * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES):(i * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES)]
    #print("Testing Block: (", (i * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES), " ", (i * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES), ")", sep="")
    if tone_power(block, len(block), SPACE_HZ, BITRATE) >= tone_power(block, len(block), MARK_HZ, BITRATE):
        space_detections += 1
    else:
        mark_detections += 1
print("Space Correlations:", space_detections, "Mark Correlations:", mark_detections)

Space Correlations: 150 Mark Correlations: 0


In [40]:
# Testing mark detection
space_detections = 0
mark_detections = 0
for i in range(len(test_space) // BYTE_LENGTH_SAMPLES):
    block = test_mark[(i * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES):(i * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES)]
    #print("Testing Block: (", (i * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES), " ", (i * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES), ")", sep="")
    if tone_power(block, len(block), MARK_HZ, BITRATE) >= tone_power(block, len(block), SPACE_HZ, BITRATE):
        mark_detections += 1
    else:
        mark_detections += 1
print("Space Correlations:", space_detections, "Mark Correlations:", mark_detections)

Space Correlations: 0 Mark Correlations: 150


In [41]:
# Correlates the bits from a given byte
def correlate_byte(block):
    bit_vals = np.zeros(8)
    for i in range(8):
        if tone_power(block[i * BIT_LENGTH_SAMPLES:(i + 1) * BIT_LENGTH_SAMPLES], BIT_LENGTH_SAMPLES, MARK_HZ, BITRATE) >= tone_power(block[i * BIT_LENGTH_SAMPLES:(i + 1) * BIT_LENGTH_SAMPLES], BIT_LENGTH_SAMPLES, SPACE_HZ, BITRATE):
            bit_vals[i] = 1
    return bit_vals

In [ ]:
# Test correlate function
print(correlate_byte(test_mark[(0 * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES):(0 * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES)]))
print(correlate_byte(test_space[(0 * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES):(0 * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES)]))

alternating_freq_test = np.zeros(8 * BIT_LENGTH_SAMPLES)
for i in range(8):
    #print((0 * BYTE_LENGTH_SAMPLES) + (i * BIT_LENGTH_SAMPLES), (0 * BYTE_LENGTH_SAMPLES) + ((i + 1) * BIT_LENGTH_SAMPLES))
    if i % 2 == 0:
        alternating_freq_test[(0 * BYTE_LENGTH_SAMPLES) + (i * BIT_LENGTH_SAMPLES):(0 * BYTE_LENGTH_SAMPLES) + ((i + 1) * BIT_LENGTH_SAMPLES)] = test_mark[(0 * BYTE_LENGTH_SAMPLES) + (i * BIT_LENGTH_SAMPLES):(0 * BYTE_LENGTH_SAMPLES) + ((i + 1) * BIT_LENGTH_SAMPLES)]
    else:
        alternating_freq_test[(0 * BYTE_LENGTH_SAMPLES) + (i * BIT_LENGTH_SAMPLES):(0 * BYTE_LENGTH_SAMPLES) + ((i + 1) * BIT_LENGTH_SAMPLES)] = test_space[(0 * BYTE_LENGTH_SAMPLES) + (i * BIT_LENGTH_SAMPLES):(0 * BYTE_LENGTH_SAMPLES) + ((i + 1) * BIT_LENGTH_SAMPLES)]
print(correlate_byte(alternating_freq_test))

[1. 1. 1. 1. 1. 1. 1. 1.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 1. 0. 1. 0. 1. 0.]


In [ ]:
# Convert a byte to its ascii value
def byte_to_ascii(data):
    b = 0
    for i in range(8):
        b += data[i] * 2 ** i
    b = int(b)
    #print(b, chr(b))
    return chr(b)

In [44]:
# Decode a file
def decode(file_data):
    s = ""
    for i in range(len(file_data) // BYTE_LENGTH_SAMPLES):
        block = file_data[(i * BYTE_LENGTH_SAMPLES) + (1 * BIT_LENGTH_SAMPLES):(i * BYTE_LENGTH_SAMPLES) + (9 * BIT_LENGTH_SAMPLES)]
        s = "".join([s, byte_to_ascii(correlate_byte(block))])
    print(s)

In [45]:
# Decode test1.wav, test2.wav, and message.wav
decode(test1_data)
decode(test2_data)
decode(message_data)

Come here, Watson, I can hear you now
What hath God wrought?
Your life will be happy and harmonious.
